# Module Import

In [1]:
!pip install -U "bitsandbytes>=0.46.1"
!pip install torch
!pip install trl
!pip install peft
!pip install peft datasets


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from  matplotlib import pylab as plt

import torch
from datasets import Dataset

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

from trl import SFTTrainer
from transformers import TrainingArguments

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [4]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


# Data Import

Based on my analysis of `dataset.csv`, you have an excellent foundation for this project. The dataset contains 32,583 rows of financial text (news headlines, tweets, and market updates) mapped to perfectly balanced sentiment labels: **10,861 positive**, **10,861 neutral**, and **10,861 negative**.

Here is how we can apply the "Industry Standard" fine-tuning methodology to
this exact dataset to build a specialized Financial Sentiment LLM.

In [5]:
# Load the dataset to inspect its structure and content
try:
    dataset = pd.read_csv("dataset.csv")

    # Get basic info
    info_buf = []
    import io
    buf = io.StringIO()
    dataset.info(buf=buf)
    info_str = buf.getvalue()

    # Get the first few rows
    head_str = dataset.head().to_string()

    print("--- INFO ---")
    print(info_str)
    print("\n--- HEAD ---")
    print(head_str)
except Exception as e:
    print(f"Error loading dataset: {e}")

--- INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32583 entries, 0 to 32582
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  32583 non-null  object
 1   text       32569 non-null  object
dtypes: object(2)
memory usage: 509.2+ KB


--- HEAD ---
  sentiment                                                                                                                  text
0  positive  All banks, lending institutions may allow a three-month moratorium on all loans, says RBI chief Shaktikanta Das ami…
1   neutral                                                                                                 Not so fast, Drake...
2  positive            FNF - ong  19.43. Trailing Stop  21.04 from 6 prior Stops of 19.71, 19.52, 18.70, 18.65, 18.07 and 17.87 -
3  positive                                      Dow opens down almost 500 points after China hits back with new round of tariffs
4  positive     

#Step 1: Data Preparation & Prompt Engineering

To train the LLM, we must convert your tabular dataset.csv into an instruction-response format. We need to drop the few rows with missing text and format the rest so the model understands the task.

In [6]:
# Load and format the data
df = dataset.copy().dropna(subset=['text'])

def generate_prompt(row):
    return f"""Below is a financial text. Analyze it and classify the sentiment as positive, neutral, or negative.

### Text:
{row['text']}

### Sentiment:
{row['sentiment']}"""

df['prompt'] = df.apply(generate_prompt, axis=1)

# Convert to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df[['prompt']])

# Create a 90/10 Train/Validation Split
split_dataset = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_data = split_dataset['train']
eval_data = split_dataset['test']

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(eval_data)}")

Training samples: 29312
Validation samples: 3257


# Step 2: Model Setup & LoRA Injection

We will load a pre-trained open-weights model in 4-bit precision to save compute, and then inject the LoRA adapters into the attention modules.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

# CHANGE THIS LINE: Use Mistral instead of LLaMA
model_id = "mistralai/Mistral-7B-v0.1"

# You don't even need the token for Mistral, but it doesn't hurt to keep it
hf_token = hf_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto", # <-- Auto run on GPU
    quantization_config=bnb_config,
    token=hf_token
)

# For Mistral, we need to add a padding token if it doesn't have one
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
print("Mistral Model loaded successfully! Ready for fine-tuning.")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

# Step 3: Supervised Fine-Tuning (SFT)

Finally, you pass your formatted dataset.csv data through the model. The base LLaMA weights stay frozen, and backpropagation only updates the small LoRA matrices based on how accurately it predicts "positive", "neutral", or "negative".

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./financial-llama",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=500 # Just a trial run
)

# Define a formatting function to combine prompt and completion
def formatting_func(example):
    # 'example' here will be a dictionary with 'prompt' and 'completion' keys
    # We return a single string for full sequence training
    return example["prompt"] + " " + example["completion"]

# Apply the formatting function to the dataset directly
# This creates a new 'text' column that SFTTrainer can use directly
train_dataset_formatted = train_data.map(
    lambda x: {"text": formatting_func(x)},
    remove_columns=train_data.column_names, # Remove original columns 'prompt' and 'completion'
    batched=False # Process one example at a time
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset_formatted, # Use the pre-formatted dataset
    eval_dataset=eval_data,
    # formatting_func=formatting_func, # No longer needed here as dataset is pre-formatted
    args=training_args,
)

trainer.train()

# Validation

In [ ]:
# Put the model in evaluation mode
model.eval()

# A hypothetical unseen text relevant to your domain
text_to_predict = "Vinamilk reports a significant year-over-year increase in Q3 Net Income, driven by highly optimized CapEx strategies."

# Format the input exactly as the model was trained, leaving the sentiment blank
prompt = f"""Below is a financial text. Analyze it and classify the sentiment as positive, neutral, or negative.

### Text:
{text_to_predict}

### Sentiment:
"""

# Tokenize and push to GPU
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generate the prediction
outputs = model.generate(**inputs, max_new_tokens=10)

# Decode and print the result
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)